# Training model with distributed learning pipeline


## Verification of dependencies and their versions

In [1]:
!pip list | grep -i kubeflow

kubeflow                  0.3.0
kubeflow_katib_api        0.19.0
kubeflow_trainer_api      2.1.0


## PyTorch DDP with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes. `TrainerClient()` verifies that you have required access to the Kubernetes cluster. Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in distributed environment.



In [2]:
import kubeflow
from kubeflow.trainer import CustomTrainer, TrainerClient
config = kubeflow.trainer.KubernetesBackendConfig()

client = TrainerClient()
trainer = TrainerClient(backend_config=config)

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob. Additionally, it might show available accelerator type and number of available resources.

In [3]:
for runtime in trainer.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

Runtime(name='deepspeed-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='deepspeed', image='ghcr.io/kubeflow/trainer/deepspeed-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='mlx-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='mlx', image='ghcr.io/kubeflow/trainer/mlx-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='retfound-pretraining-configurable', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='ghcr.io/andesterson/dinov3_gr_pretrain-runner:v0.0.1', num_nodes=1, device='gpu', device_count='4'), pretrained_model=None)
Runtime(name='retfound-pretraining-gram', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='ghcr.io/andesterson/dinov3_gr_pr

## Run the Distributed TrainJob

Kubeflow TrainJob will train the above model on PyTorch nodes defined by `NUM_NODES` and each node with `RESOURCES_PER_NODE`.

In [4]:
from kubeflow.trainer import TrainerClient, CustomTrainer, options
from kubeflow.trainer.options import TrainerCommand
from kubeflow.trainer.types.types import CustomTrainerContainer

# ------------------------------------------------
# Configuration of resources with default notebeook namespace (2CPU;4GiMEM)
# ------------------------------------------------

## Set how many PyTorch nodes you want to use for distributed training.
NUM_NODES = 1

# Set the resources for each PyTorch node.
RESOURCES_PER_NODE = {
    "cpu": "4",           # CPUs per node
    "memory": "64Gi",     # Memory in GiB per node (tried 2Gi CrashLoopBackOff/OOMKilled), 64Gi works
    "nvidia.com/gpu": 1,  # GPUs per node (the number will depend on the available resources)
}

GITHUB_CONTAINER_REGISTRY = "ghcr.io/xfetus/fetal-ultrasound-edm2/fetal-ultrasound-edm2-distributed-learning:v0.0.0"
# VERSION_ID=v0.0.0 #FROM docker.io/pytorch/pytorch:2.9.1-cuda12.8-cudnn9-devel / RUN mkdir -p /workspace && chmod -R 777 /workspace


# ------------------------------------------------
# Seeting up to mount scratch-volume
# ------------------------------------------------
volume_name = "scratch-volume"
pvc_name = "scratch-volume"
mount_scrath_path = "/scratch-volume"

pod_volumes = [
    {
        "name": volume_name,
        "persistentVolumeClaim": {
            "claimName": pvc_name
        }
    }
]
volume_mounts = [
    {
        "name": volume_name,
        "mountPath": mount_scrath_path
    }
]
container_override = options.ContainerOverride(
    name="node",
    volume_mounts=volume_mounts
)
pod_spec_override = options.PodSpecOverride(
    volumes=pod_volumes,
    containers=[container_override]
)
pod_template_override = options.PodTemplateOverride(
    target_jobs=["node"],
    spec=pod_spec_override
)
pod_template_overrides = options.PodTemplateOverrides(
    pod_template_override
)


ENV_VARS = {
    "MASTER_ADDR": "localhost",
    "MASTER_PORT": "12355",
    "WORLD_SIZE": "1",
    "RANK": "0",
    "LOCAL_RANK": "0",
}

command = TrainerCommand(
    command=[
        "torchrun",
        "/scratch-volume/fetal-ultrasound-edm2/train_edm2.py",
        "--outdir", "/scratch-volume/FETAL_PLANES_DB/OUTPUT_DIRECTORY", # pragma: allowlist secret
        "--data", "/scratch-volume/FETAL_PLANES_DB", # pragma: allowlist secret
        "--batch", "4",
        "--preset", "edm2-img512-s",
        "--batch-gpu", "4",
    ]
)


#NOTES
## torchrun commands for distributed learning but for this notebook resource are injected by Kubeflow trainer
        # "--standalone", #Single-node multi-worker # pragma: allowlist secret
        # f"--nnodes={NUM_NODES}",
        # f"--nproc_per_node={NPROC_PER_NODE}",#set at by Kubeflow trainer level
        # f"--node_rank={NODE_RANK}", #injected by Kubeflow trainer
        # f"--master_addr={MASTER_ADDR}",#injected by Kubeflow trainer
        # f"--master_port={MASTER_PORT}",#injected by Kubeflow trainer

## the following settings show some history of what has been tested
#(1/n: works)--------------------------------------
        # "--batch", "1",
        # "--preset", "edm2-img512-xxs",
        # "--batch-gpu", "1",

#(2/n: timeout)--------------------------------------
        # --batch 8 \
        # --preset edm2-img512-s \
        # --batch-gpu=8
### createse External termination (SIGTERM)  Most likely: timeout limit


In [5]:
job_id = trainer.train(
    runtime=torch_runtime,
    trainer=CustomTrainerContainer(
        image=GITHUB_CONTAINER_REGISTRY,
        num_nodes=NUM_NODES,
        resources_per_node=RESOURCES_PER_NODE,
        env=ENV_VARS        
    ),
    options=[command, pod_template_overrides],
)

In [6]:
#Check job status directly
job = trainer.get_job(job_id)
print(f"\nJob ID: {job_id}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")


Job ID: n04d9e4317ee
Job Status: Created
Creation Time: 2026-04-24 23:57:41+00:00

Job details: TrainJob(name='n04d9e4317ee', runtime=Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None), steps=[], num_nodes=1, creation_timestamp=datetime.datetime(2026, 4, 24, 23, 57, 41, tzinfo=TzInfo(0)), status='Created')


In [ ]:
from datetime import datetime
import time
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(trainer.get_job_logs(job_id, follow=True))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)


### EXPECTED TERMINAL LOG FROM LOCAL MACHINE:
#### COMMAND
# torchrun --standalone --nproc_per_node=1 train_edm2.py 
# --outdir ~/scratch-volume/FETAL_PLANES_DB/OUTPUT_DIRECTORY 
# --data ~/scratch-volume/FETAL_PLANES_DB             
# --batch 1         
# --preset edm2-img512-xxs
# --batch-gpu=1

#### LOGS
# Output directory:        /home/mxochicale/scratch-volume/FETAL_PLANES_DB/OUTPUT_DIRECTORY
# Dataset path:            /home/mxochicale/scratch-volume/FETAL_PLANES_DB
# Class-conditional:       True
# Number of GPUs:          1
# Batch size:              8
# Mixed-precision:         True

# Loading dataset...
# Dataset size: 7129
# Setting up encoder...
# Constructing network...
# {'class_name': 'training.networks_edm2.Precond', 'model_channels': 192, 'dropout': 0.0, 'use_fp16': True} {'img_resolution': 64, 'img_channels': 4, 'label_dim': 6}

# Precond                Parameters  Buffers  Output shape      Datatype
# ---                    ---         ---      ---               ---     
# unet.emb_fourier       -           384      [8, 192]          float32 
# unet.emb_noise         147456      -        [8, 768]          float32 
# unet.emb_label         4608        -        [8, 768]          float32 
# ...
# unet.dec.64x64_block3  1216513     -        [8, 192, 64, 64]  float16 
# unet.out_conv          6912        -        [8, 4, 64, 64]    float16 
# unet                   1           -        [8, 4, 64, 64]    float16 
# <top-level>            128         256      [8, 4, 64, 64]    float32 
# ---                    ---         ---      ---               ---     
# Total                  279449445   640      -                 -       
#
# Setting up training state...
# Training from 0 kimg to 2147483 kimg
#
# Setting up validation set...
# val dataset size: 5271
#   Processing 5271/5271 ... Pre-encoding validation latents... 
# Validation: 5271 images | sigma [0.019, 28.501]
#   0%|          | 0/131072 [00:00<?, ?it/s]NETWORK: Precond(
#   (unet): UNet(
#     (emb_fourier): MPFourier()
#     (emb_noise): MPConv()
#     (emb_label): MPConv()
#     (enc): ModuleDict(
#       (64x64_conv): MPConv()
#       (64x64_block0): Block(
#         (conv_res0): MPConv()
#         (emb_linear): MPConv()
#         (conv_res1): MPConv()
#       )
#     ...
#       (64x64_block3): Block(
#         (conv_res0): MPConv()
#         (emb_linear): MPConv()
#         (conv_res1): MPConv()
#         (conv_skip): MPConv()
#       )
#     )
#     (out_conv): MPConv()
#   )
#   (logvar_fourier): MPFourier()
#   (logvar_linear): MPConv()
# )
# --------------------
#   0%|          | 0/131072 [00:00<?, ?it/s]Status: kimg 0.0       time 30m 42s      
#                                                                  sec/tick 1841.65  sec/kimg 0.000   maintenance 1841.65 cpumem 4.18   gpumem 2.09   reserved 2.86  
# Val_loss:  1.5974
# Saving network-snapshot-0000000-0.050.pkl ... done
# Saving network-snapshot-0000000-0.100.pkl ... done
#  18%|#8        | 23843/131072 [2:05:01<9:22:16,  3.18it/s]    
# ...
#

Waiting for job logs...
[23:57:41] Waiting... (1s)
[23:57:42] Waiting... (2s)
[23:57:43] Waiting... (3s)
[23:57:44] Waiting... (4s)
[23:57:45] Waiting... (5s)
[23:57:46] Waiting... (6s)
[23:57:48] Waiting... (7s)
[23:57:49] Waiting... (8s)
[23:57:50] Waiting... (9s)
[23:57:51] Waiting... (10s)
[23:57:52] Waiting... (11s)
[23:57:53] Waiting... (12s)
[23:57:54] Waiting... (13s)
[23:57:55] Waiting... (14s)
[23:57:56] Waiting... (15s)
[23:57:57] Waiting... (16s)
[23:57:58] Waiting... (17s)
[23:57:59] Waiting... (18s)
[23:58:00] Waiting... (19s)
[23:58:01] Waiting... (20s)
[23:58:02] Waiting... (21s)
[23:58:03] Waiting... (22s)
[23:58:04] Waiting... (23s)
[23:58:05] Waiting... (24s)
[23:58:06] Waiting... (25s)
[23:58:07] Waiting... (26s)
[23:58:08] Waiting... (27s)
[23:58:09] Waiting... (28s)
[23:58:10] Waiting... (29s)
[23:58:11] Waiting... (30s)
[23:58:12] Waiting... (31s)
[23:58:13] Waiting... (32s)
[23:58:14] Waiting... (33s)
[23:58:15] Waiting... (34s)
[23:58:16] Waiting... (35s)
[23:5